# Practical LLM lab — numerical workbook

## Goal

Explain a training step before downloading a model. Eight small exercises connect
causal token likelihoods, LoRA, quantisation, memory, DPO, and GRPO using explicit
numbers. **Predict each answer before running the reference cell.** Change the
numbers and explain the result afterwards.

All computations use local CPU tensors. No pretrained model is downloaded or trained
by this workbook. SFT/DPO helper calls use the real [training core](llm_lab/training.py);
quantisation uses the real [numerical module](llm_lab/quantisation.py). GRPO cells are
formula demonstrations; this lab has no full pretrained-model GRPO training command.

Read [LLMs as policies](lessons/01-llms-as-policies.md),
[LoRA training](lessons/02-real-lora-training.md),
[quantisation/QLoRA](lessons/03-quantisation-and-qlora.md), and
[preferences and rewards](lessons/04-preferences-and-rewards.md) alongside the exercises.

## Setup

Use the Python environment from [README.md](README.md). Run all cells in order;
self-check code is initially collapsed in frontends supporting Jupyter's source-hidden
metadata. Written answers also appear in expandable sections.
All data below are constructed examples, not experimental model results.

In [ ]:
from pathlib import Path
import sys
import math
import platform
import numpy as np
import torch

candidates = [Path.cwd(), Path.cwd() / "llm-lab", Path.cwd().parent / "llm-lab"]
lab_root = next((p for p in candidates if (p / "llm_lab/quantisation.py").exists()), None)
if lab_root is None:
    raise RuntimeError("Run the notebook from the repository root or llm-lab directory")
if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))
from llm_lab.quantisation import symmetric_quantize, affine_quantize, estimate_lora_memory
from llm_lab.training import completion_logprobs, dpo_losses

torch.manual_seed(0)
np.random.seed(0)
print(f"CPU tutorial | Python {platform.python_version()} | PyTorch {torch.__version__} | NumPy {np.__version__}")

## Steps

### Exercise 1 — which token predictions contribute to SFT?

A causal model's logits at position $t$ predict the token at $t+1$.
The labels `[-100, -100, 1, 2, -100]` mark prompt/padding positions with `-100`.
Only the two completion targets, tokens 1 and 2, should be scored.
Their probabilities are 0.8 and 0.7.

**Predict:** what are the sequence log-probability and mean token NLL? Does changing
a prompt-only prediction change either? Here log means the natural logarithm.

In [ ]:
probabilities = torch.tensor([[
    [0.20, 0.30, 0.50],  # predicts a masked prompt token
    [0.10, 0.80, 0.10],  # predicts completion target 1
    [0.10, 0.20, 0.70],  # predicts completion target 2
    [0.90, 0.05, 0.05],  # predicts masked padding
    [0.30, 0.30, 0.40],  # final logits have no next label here
]], dtype=torch.float64)
labels = torch.tensor([[-100, -100, 1, 2, -100]])
logits = probabilities.log()
sequence_logp, target_counts = completion_logprobs(logits, labels)
mean_token_nll = -sequence_logp.sum() / target_counts.sum()
print({"completion_logp": sequence_logp.item(), "target_tokens": target_counts.item(),
       "mean_token_nll": mean_token_nll.item()})

<details><summary>Answer and transfer check</summary>

The sequence probability is $0.8(0.7)=0.56$, log-probability is
$\log0.56=-0.5798185$, and mean token NLL is $0.5798185/2=0.2899092$.
The denominator counts completion targets, not padded sequence length.
A prompt token can still affect future predictions as input even though the model
is not trained to predict that token in this completion-only objective.
</details>

In [ ]:
assert target_counts.tolist() == [2]
assert math.isclose(sequence_logp.item(), math.log(0.56), abs_tol=1e-6)
altered = logits.clone()
altered[:, 0, :] = torch.tensor([10., -10., 0.])
assert torch.allclose(completion_logprobs(altered, labels)[0], sequence_logp)
print("SFT masking and causal shift checks passed")

### Exercise 2 — LoRA is a second matrix path

Use $y=W_0x+(\alpha/r)B(Ax)$ with rank $r=1$, $\alpha=2$,
$W_0=I_2$, $A=[1,-1]$, $B=[0.2,0.4]^T$, and $x=[3,1]^T$.

**Predict:** calculate `A @ x` first, then the scaled adapter output. How many
parameters are trainable? Does rank one save parameters for this tiny matrix?

In [ ]:
base_weight = torch.eye(2, dtype=torch.float64)
lora_a = torch.tensor([[1., -1.]], dtype=torch.float64)
lora_b = torch.tensor([[0.2], [0.4]], dtype=torch.float64)
input_vector = torch.tensor([3., 1.], dtype=torch.float64)
rank, alpha = 1, 2
base_output = base_weight @ input_vector
adapter_output = (alpha / rank) * lora_b @ (lora_a @ input_vector)
combined_output = base_output + adapter_output
print({"base": base_output.tolist(), "adapter": adapter_output.tolist(),
       "combined": combined_output.tolist(), "trainable_parameters": lora_a.numel() + lora_b.numel()})
large_base_parameters = 1024 * 1024
large_adapter_parameters = 8 * (1024 + 1024)
print({"1024_square_rank8_fraction": large_adapter_parameters / large_base_parameters})

<details><summary>Answer and transfer check</summary>

$Ax=2$, scaled adapter output is $(0.8,1.6)$, and combined output is $(3.8,2.6)$.
The small example trains four parameters, equal to its four base parameters; small
rank alone does not promise savings. At width 1024 and rank 8, the adapter trains
16,384 parameters versus 1,048,576 base parameters, a fraction $1/64=1.5625\%$.
The base still occupies memory even though its parameters are frozen.
</details>

In [ ]:
torch.testing.assert_close(combined_output, torch.tensor([3.8, 2.6], dtype=torch.float64))
merged = base_weight + (alpha / rank) * lora_b @ lora_a
torch.testing.assert_close(merged @ input_vector, combined_output)
assert large_adapter_parameters == 16_384
print("LoRA path, shapes, merge identity, and parameter count passed")

### Exercise 3 — four bits of storage, not four-bit floating computation

Quantise $(-1,-0.6,-0.1,0,0.3,0.8,1)$ symmetrically at four bits.
This module uses signed levels -7 through 7, scale $1/7$, and stores shifted codes
0 through 14. Rounding is nearest with ties to even.

**Predict:** what signed integer represents 0.8? How many payload bytes store seven
four-bit codes? Is that the full Python object's memory usage?

In [ ]:
weights = torch.tensor([-1., -.6, -.1, 0., .3, .8, 1.], dtype=torch.float64)
quantized = symmetric_quantize(weights, bits=4)
reconstructed = quantized.dequantize()
print({"scale": quantized.scale, "integer_codes": quantized.unpack().tolist(),
       "signed_levels": (quantized.unpack().to(torch.int16) - quantized.zero_point).tolist(),
       "reconstructed": [round(v, 6) for v in reconstructed.tolist()],
       "packed_bytes": quantized.payload_nbytes,
       "max_abs_error": (weights - reconstructed).abs().max().item()})

<details><summary>Answer and transfer check</summary>

$0.8/(1/7)=5.6$ rounds to signed level 6, reconstructing $6/7=0.857143$.
Seven codes occupy $\lceil28/8\rceil=4$ payload bytes, with padding in the final byte.
Scale/zero point, shape, Python metadata, original weights, and any reconstructed
float tensor are additional storage. This is uniform quantisation, not NF4.
</details>

In [ ]:
assert quantized.payload.dtype == torch.uint8
assert quantized.payload_nbytes == 4
assert (weights - reconstructed).abs().max() <= quantized.scale / 2 + 1e-12
print("Packed storage and half-step error bound passed")

### Exercise 4 — clipping and an all-zero tensor

Use two-bit affine quantisation with calibration range [-1, 2]. Then $s=1$,
zero point $z=1$, and codes 0–3 represent -1, 0, 1, 2.

**Predict:** where does an input 4 land, and does its error obey the half-step bound?
What should happen to an all-zero tensor if automatic calibration would give scale 0?

In [ ]:
affine_input = torch.tensor([-1., -.4, 0., .6, 2., 4.], dtype=torch.float64)
affine_result = affine_quantize(affine_input, bits=2, value_range=(-1., 2.))
zero_result = symmetric_quantize(torch.zeros(3, 5), bits=4)
print({"affine_codes": affine_result.unpack().tolist(),
       "affine_reconstructed": affine_result.dequantize().tolist(),
       "outlier_error": abs(4 - affine_result.dequantize()[-1].item()),
       "zero_scale_convention": zero_result.scale,
       "zero_nonzero_reconstructions": torch.count_nonzero(zero_result.dequantize()).item()})

<details><summary>Answer and transfer check</summary>

The outlier saturates to code 3, reconstructing 2 with error 2, greater than $s/2=0.5$.
The nearest-rounding bound applies without saturation. The all-zero convention is
scale 1 and exact zero reconstruction; no divide-by-zero is needed.
</details>

In [ ]:
assert affine_result.unpack().tolist() == [0, 1, 1, 2, 3, 3]
assert zero_result.scale == 1 and torch.count_nonzero(zero_result.dequantize()) == 0
assert abs(4 - affine_result.dequantize()[-1].item()) == 2
print("Saturation and zero-scale handling passed")

### Exercise 5 — a weight budget is not peak training memory

Assume one million frozen base parameters, 10,000 fp32 adapter parameters,
fp32 adapter gradients and two fp32 Adam moments. For quantised bases, assume
one fp32 scale per 64 weights, without double quantisation.

**Predict:** count adapter training-state bytes before counting the base. What does
four-bit base storage save? Which major memory components are missing?

In [ ]:
memory_by_bits = {bits: estimate_lora_memory(1_000_000, 10_000, base_bits=bits)
                  for bits in (32, 16, 8, 4)}
for bits, parts in memory_by_bits.items():
    print(f"{bits:2d}-bit base: {parts['accounted_bytes']:,} accounted bytes")
kv_cache_bytes = 2 * 12 * 1 * 1024 * 4 * 64 * 2
print(f"Example fp16 KV cache: {kv_cache_bytes:,} bytes = {kv_cache_bytes / 2**20:g} MiB")

<details><summary>Answer and transfer check</summary>

Each fp32 trainable parameter costs $4+4+8=16$ bytes under these assumptions,
so adapter training states total 160,000 bytes. The four-bit base payload is
500,000 bytes; scales add 62,500, for 722,500 accounted bytes. The fp32-base
comparison totals 4,160,000 bytes. The example KV cache is 12 MiB, already
larger than either miniature weight budget. Activations, workspaces, reference
models, unquantised modules, and transient buffers are still missing.
</details>

In [ ]:
assert memory_by_bits[4]["accounted_bytes"] == 722_500
assert memory_by_bits[32]["accounted_bytes"] == 4_160_000
assert kv_cache_bytes == 12 * 2**20
print("Memory arithmetic passed; these remain estimates, not device measurements")

### Exercise 6 — DPO rewards a relative improvement in preference odds

Let the current policy assign completion probabilities 0.6 to chosen and 0.2 to
rejected; the fixed reference assigns 0.4 to each. Remaining probability may belong
to other completions. Define

$$m=(\log\pi_w-\log\pi_l)-(\log\pi_{ref,w}-\log\pi_{ref,l}),\qquad
L=-\log\sigma(\beta m),\quad\beta=0.5.$$

**Predict:** does the loss fall below $\log2$? What are the signs of the derivatives
with respect to the chosen and rejected log-probabilities?

In [ ]:
chosen_logp = torch.tensor([math.log(.6)], dtype=torch.float64, requires_grad=True)
rejected_logp = torch.tensor([math.log(.2)], dtype=torch.float64, requires_grad=True)
reference_chosen = torch.tensor([math.log(.4)], dtype=torch.float64, requires_grad=True)
reference_rejected = torch.tensor([math.log(.4)], dtype=torch.float64, requires_grad=True)
losses, margins = dpo_losses(chosen_logp, rejected_logp, reference_chosen, reference_rejected, beta=.5)
losses.mean().backward()
print({"margin": margins.item(), "loss": losses.item(),
       "dL_dchosen_logp": chosen_logp.grad.item(),
       "dL_drejected_logp": rejected_logp.grad.item(),
       "reference_gradient": reference_chosen.grad})

<details><summary>Answer and transfer check</summary>

$m=\log3=1.098612$, and loss is $\log(1+1/\sqrt3)=0.455746<\log2$.
The chosen log-probability derivative is $-0.5/(1+\sqrt3)=-0.183013$;
the rejected derivative is positive 0.183013. Gradient descent therefore pushes
chosen upward relative to rejected. These leaf log-probabilities illustrate partial
derivatives; a real model couples probabilities through its shared parameters.
Reference scores receive no gradient and must remain fixed across training updates.
</details>

In [ ]:
assert math.isclose(margins.item(), math.log(3), abs_tol=1e-12)
assert math.isclose(losses.item(), math.log1p(1/math.sqrt(3)), abs_tol=1e-12)
assert chosen_logp.grad.item() < 0 < rejected_logp.grad.item()
assert reference_chosen.grad is None and reference_rejected.grad is None
initial_loss, initial_margin = dpo_losses(chosen_logp.detach(), rejected_logp.detach(),
                                       chosen_logp.detach(), rejected_logp.detach(), beta=.5)
assert math.isclose(initial_loss.item(), math.log(2), abs_tol=1e-12)
print("Real DPO helper, gradient direction, frozen reference, and initial loss passed")

### Exercise 7 — GRPO's group baseline and a group with no signal

For four completions of the **same prompt**, rewards are $(0,1,1,2)$.
Use $A_i=(r_i-\bar r)/(\sigma_r+\epsilon)$ with population standard deviation
(`unbiased=False`) and $\epsilon=10^{-8}$.

**Predict:** which completions should be reinforced, discouraged, or unchanged by
the reward term? What happens if all rewards are one? Distinguish this small
normalisation example from a complete GRPO objective and rollout implementation.

In [ ]:
group_rewards = torch.tensor([0., 1., 1., 2.], dtype=torch.float64)
def group_advantages(rewards):
    return (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-8)
advantages = group_advantages(group_rewards)
uniform_advantages = group_advantages(torch.ones(4, dtype=torch.float64))
print({"mean_reward": group_rewards.mean().item(),
       "population_std": group_rewards.std(unbiased=False).item(),
       "advantages": advantages.tolist(),
       "equal_reward_advantages": uniform_advantages.tolist()})

<details><summary>Answer and transfer check</summary>

Mean reward is 1, variance is $[(0-1)^2+0+0+(2-1)^2]/4=0.5$, and standard
deviation is $1/\sqrt2$. Advantages are approximately $(-\sqrt2,0,0,\sqrt2)$.
Equal rewards give all-zero advantages, so that group supplies no reward gradient;
a separate KL term may still act. Subtracting a group mean also means the first
sample contributes to its own baseline; do not treat this as an automatically
unbiased state-only baseline theorem. A leave-one-out baseline is a different estimator.
</details>

In [ ]:
torch.testing.assert_close(advantages, torch.tensor([-math.sqrt(2), 0, 0, math.sqrt(2)], dtype=torch.float64), atol=3e-8, rtol=0)
assert uniform_advantages.tolist() == [0., 0., 0., 0.]
torch.testing.assert_close(group_advantages(group_rewards + 10), advantages)
print("Group statistics, constant-reward edge, and shift invariance passed")

### Exercise 8 — clipping is sign-sensitive

Consider the PPO-style reward surrogate used in many GRPO implementations:
$\min(\rho A,\operatorname{clip}(\rho,1-\epsilon,1+\epsilon)A)$, with
ratio $\rho=\pi_{new}(a\mid s)/\pi_{old}(a\mid s)$ and $\epsilon=0.2$.
Hold advantages fixed for this derivative calculation.

**Predict:** compare $(A,\rho)=(1,1.5)$, $(-1,1.5)$ and $(-1,0.5)$.
Does clipping simply cap every gradient whenever the ratio leaves the interval?

In [ ]:
fixed_advantages = torch.tensor([1., -1., -1.], dtype=torch.float64)
ratios = torch.tensor([1.5, 1.5, .5], dtype=torch.float64, requires_grad=True)
surrogate = torch.minimum(ratios * fixed_advantages,
                          ratios.clamp(.8, 1.2) * fixed_advantages)
surrogate.sum().backward()
print({"surrogate": surrogate.tolist(), "derivative_wrt_ratio": ratios.grad.tolist()})

<details><summary>Answer and transfer check</summary>

The surrogate is $(1.2,-1.5,-0.8)$, with ratio derivatives $(0,-1,0)$.
For a positive advantage, excessive probability increases stop earning extra
surrogate reward. For a negative advantage, a harmful probability increase still
receives a correcting gradient. Clipping this surrogate is not a hard bound on
all actual policy changes or on KL after optimisation.
</details>

In [ ]:
torch.testing.assert_close(surrogate, torch.tensor([1.2, -1.5, -.8], dtype=torch.float64))
torch.testing.assert_close(ratios.grad, torch.tensor([0., -1., 0.], dtype=torch.float64))
print("Clipped-surrogate values and derivatives passed")

## Checks

Running top-to-bottom verifies:

- The same causal completion likelihood and DPO functions used in the real trainer.
- LoRA's forward/merge identity and parameter-count arithmetic.
- Actual packed storage, reconstruction error, clipping, and zero handling.
- Explicit memory assumptions, fixed-reference gradients, group edge cases, and clipping signs.

These checks validate numerical mechanisms. They do not establish pretrained-model
learning improvement, GPU compatibility, model safety, or a successful online GRPO run.

## Next Steps

1. Follow [Running a training project](lessons/05-running-a-training-project.md) and
   [README.md](README.md) to evaluate the initial model, train SFT, save/reload its
   adapter, then train DPO with a fixed initial-policy reference.
2. On the RTX 5090, run the kernel checks in [NVIDIA.md](NVIDIA.md) before NF4 training.
   CPU/MPS LoRA and the uniform quantisation examples are different execution paths.
3. Read [week 20's guide](../week-20-grpo-rloo-rlvr/teaching_guide.md) for the online
   reward/rollout context omitted from these tiny GRPO calculations.
4. Change one assumption per exercise, predict the outcome, and keep a short record
   of discrepancies. A correct explanation should survive a new numerical example.